# Renommage des dossiers par pays

Ce notebook parcourt les dossiers dans `../data`, lit les coordonnées lat/lon depuis `output_data_points.csv`, effectue un geocodage inverse pour obtenir le nom du pays, puis renomme le dossier.

In [3]:
%pip install reverse_geocoder pandas pycountry -q


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
import shutil
import pandas as pd
import reverse_geocoder as rg
from pathlib import Path

# Correspondance code pays ISO -> nom complet
import pycountry

def get_country_name(lat, lon):
    """Retourne le nom complet du pays a partir des coordonnees."""
    results = rg.search((lat, lon), verbose=False)
    if not results:
        return None
    cc = results[0].get('cc', '')
    try:
        country = pycountry.countries.get(alpha_2=cc)
        return country.name if country else cc
    except Exception:
        return cc

In [ ]:
%pip install pycountry -q

In [5]:
data_dir = Path('../data')

# Collecter tous les sous-dossiers directs (pas les fichiers)
subdirs = [d for d in data_dir.iterdir() if d.is_dir()]
print(f"{len(subdirs)} dossier(s) trouve(s) dans {data_dir.resolve()}")
for d in subdirs:
    print(f"  - {d.name}")

62 dossier(s) trouve(s) dans /home/agbelgaid/Documents/WORKSPACE/DataCollection/SoilHive/data
  - Y29uaXRlLmdib2RvZ2JlQHVtNnAubWE=_1771261975584
  - Y29uaXRlLmdib2RvZ2JlQHVtNnAubWE=_1771261913876
  - Y29uaXRlLmdib2RvZ2JlQHVtNnAubWE=_1771261647952
  - Y29uaXRlLmdib2RvZ2JlQHVtNnAubWE=_1771261743087
  - ZHNjb25pdGVAZ21haWwuY29t_1771515543832
  - ZHNjb25pdGVAZ21haWwuY29t_1771515392320
  - ZHNjb25pdGVAZ21haWwuY29t_1771532782319
  - Y29uaXRlLmdib2RvZ2JlQHVtNnAubWE=_1771261955340
  - ZHNjb25pdGVAZ21haWwuY29t_1771532729389
  - Y29uaXRlLmdib2RvZ2JlQHVtNnAubWE=_1771261822035
  - Y29uaXRlLmdib2RvZ2JlQHVtNnAubWE=_1771261733250
  - Y29uaXRlLmdib2RvZ2JlQHVtNnAubWE=_1771261784038
  - Y29uaXRlLmdib2RvZ2JlQHVtNnAubWE=_1771261639137
  - Y29uaXRlLmdib2RvZ2JlQHVtNnAubWE=_1771261713763
  - Y29uaXRlLmdib2RvZ2JlQHVtNnAubWE=_1771261793437
  - ZHNjb25pdGVAZ21haWwuY29t_1771515433645
  - Y29uaXRlLmdib2RvZ2JlQHVtNnAubWE=_1771261665879
  - ZHNjb25pdGVAZ21haWwuY29t_1771515448464
  - ZHNjb25pdGVAZ21haWwuY29t_1771532

In [6]:
rename_plan = []  # liste de (ancien_chemin, nouveau_chemin, pays)

for folder in subdirs:
    csv_path = folder / 'output_data_points.csv'

    if not csv_path.exists():
        print(f"[SKIP] Pas de CSV dans : {folder.name}")
        continue

    # Lire la premiere ligne de donnees
    try:
        df = pd.read_csv(csv_path, nrows=1)
        lat = df['lat'].iloc[0]
        lon = df['lon'].iloc[0]
    except Exception as e:
        print(f"[ERREUR] Lecture CSV {folder.name} : {e}")
        continue

    country = get_country_name(lat, lon)
    if not country:
        print(f"[SKIP] Pays introuvable pour lat={lat}, lon={lon} dans {folder.name}")
        continue

    # Nettoyer le nom pour un usage comme nom de dossier
    safe_name = country.replace(' ', '_').replace('/', '-')

    # Gerer les doublons : ajouter un suffixe si necessaire
    new_path = data_dir / safe_name
    suffix = 1
    while new_path.exists() and new_path != folder:
        new_path = data_dir / f"{safe_name}_{suffix}"
        suffix += 1

    rename_plan.append((folder, new_path, country, lat, lon))
    print(f"  {folder.name}")
    print(f"  -> {new_path.name}  (lat={lat:.4f}, lon={lon:.4f}, pays={country})")
    print()

print(f"\n{len(rename_plan)} dossier(s) a renommer.")

  Y29uaXRlLmdib2RvZ2JlQHVtNnAubWE=_1771261975584
  -> Slovenia  (lat=46.6823, lon=15.9975, pays=Slovenia)

  Y29uaXRlLmdib2RvZ2JlQHVtNnAubWE=_1771261913876
  -> Portugal  (lat=32.7069, lon=-16.7878, pays=Portugal)

  Y29uaXRlLmdib2RvZ2JlQHVtNnAubWE=_1771261647952
  -> Belarus  (lat=53.5200, lon=29.5200, pays=Belarus)

  Y29uaXRlLmdib2RvZ2JlQHVtNnAubWE=_1771261743087
  -> French_Guiana  (lat=4.3297, lon=-51.9700, pays=French Guiana)

  ZHNjb25pdGVAZ21haWwuY29t_1771515543832
  -> Honduras  (lat=15.7854, lon=-87.5957, pays=Honduras)

  ZHNjb25pdGVAZ21haWwuY29t_1771515392320
  -> Peru  (lat=-5.7667, lon=-76.0833, pays=Peru)

  ZHNjb25pdGVAZ21haWwuY29t_1771532782319
  -> Thailand  (lat=20.4431, lon=99.9033, pays=Thailand)

  Y29uaXRlLmdib2RvZ2JlQHVtNnAubWE=_1771261955340
  -> Serbia  (lat=43.3872, lon=19.6233, pays=Serbia)

  ZHNjb25pdGVAZ21haWwuY29t_1771532729389
  -> Philippines  (lat=10.0430, lon=118.7307, pays=Philippines)

  Y29uaXRlLmdib2RvZ2JlQHVtNnAubWE=_1771261822035
  -> Lithuania

In [7]:
# Renommage effectif
for old_path, new_path, country, lat, lon in rename_plan:
    if old_path == new_path:
        print(f"[INCHANGE] {old_path.name} (deja nomme correctement)")
        continue
    try:
        old_path.rename(new_path)
        print(f"[OK] {old_path.name} -> {new_path.name}")
    except Exception as e:
        print(f"[ERREUR] {old_path.name} : {e}")

print("\nTermine.")

[OK] Y29uaXRlLmdib2RvZ2JlQHVtNnAubWE=_1771261975584 -> Slovenia
[OK] Y29uaXRlLmdib2RvZ2JlQHVtNnAubWE=_1771261913876 -> Portugal
[OK] Y29uaXRlLmdib2RvZ2JlQHVtNnAubWE=_1771261647952 -> Belarus
[OK] Y29uaXRlLmdib2RvZ2JlQHVtNnAubWE=_1771261743087 -> French_Guiana
[OK] ZHNjb25pdGVAZ21haWwuY29t_1771515543832 -> Honduras
[OK] ZHNjb25pdGVAZ21haWwuY29t_1771515392320 -> Peru
[OK] ZHNjb25pdGVAZ21haWwuY29t_1771532782319 -> Thailand
[OK] Y29uaXRlLmdib2RvZ2JlQHVtNnAubWE=_1771261955340 -> Serbia
[OK] ZHNjb25pdGVAZ21haWwuY29t_1771532729389 -> Philippines
[OK] Y29uaXRlLmdib2RvZ2JlQHVtNnAubWE=_1771261822035 -> Lithuania
[OK] Y29uaXRlLmdib2RvZ2JlQHVtNnAubWE=_1771261733250 -> Finland
[OK] Y29uaXRlLmdib2RvZ2JlQHVtNnAubWE=_1771261784038 -> Ireland
[OK] Y29uaXRlLmdib2RvZ2JlQHVtNnAubWE=_1771261639137 -> Austria
[OK] Y29uaXRlLmdib2RvZ2JlQHVtNnAubWE=_1771261713763 -> Denmark
[OK] Y29uaXRlLmdib2RvZ2JlQHVtNnAubWE=_1771261793437 -> Italy
[OK] ZHNjb25pdGVAZ21haWwuY29t_1771515433645 -> Bolivia,_Plurinational_State_o